# Logical-Form-Guided Hybrid Reasoning

*Level 9 — Knowledge-Augmented Generation (KAG)*

## Objective

The third piece: a question is first parsed into a **logical form** -- which of four operators
(`retrieval`, `kg_reasoning`, `language_reasoning`, `numerical_calculation`) it actually needs --
then the router dispatches to just those operators, merges whatever evidence they produce, and
`language_reasoning` always makes the final yes/no/maybe call.

This notebook runs the parser and the full pipeline on real PubMedQA questions, plus the two
hand-authored example questions from this level's README that exercise the KG-reasoning and
numerical operators directly -- real PubMedQA questions are always yes/no/maybe about a study's
own findings, never "which study is largest", so those two operators need a hand-built example
to demonstrate at all (the same precedent 04-adaptive-rag set for question types a real dataset
doesn't naturally contain).

In [1]:
import sys
from pathlib import Path

LEVEL_DIR = Path.cwd().parent
sys.path.insert(0, str(LEVEL_DIR))
sys.path.insert(0, str(LEVEL_DIR / "reasoning-engine"))

from kag_common.dataset import prepare
from kag_common.llm import OllamaLLM
from kag_common.embed import OllamaEmbedder, embed_texts
from indexing.graph_builder import build_graph
from logical_form_parser import parse_logical_form
from operator_router import answer_question

data = prepare(n_documents=10, seed=11)
llm = OllamaLLM()
embedder = OllamaEmbedder()

graph, validator, mutual_index = build_graph(data.corpus, llm)
doc_ids, matrix = embed_texts(data.corpus, embedder, cache_name="nb03_corpus")
print(f"Graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")
print("Validator:", validator.summary())

Graph: 33 nodes, 31 edges
Validator: {'accepted_entities': 39, 'rejected_entities': 0, 'entity_rejection_rate': 0.0, 'accepted_relations': 31, 'rejected_relations': 5, 'relation_rejection_rate': 0.1388888888888889}


## Parsing real and hand-authored questions into logical forms

In [2]:
real_question = next(iter(data.questions.values()))["question"]

example_questions = [
    real_question,
    "Was the intervention studied in a population larger than 500 patients?",
    "What outcome was reported for the largest study of this condition?",
]

for q in example_questions:
    form = parse_logical_form(q, llm)
    print(f"Q: {q}")
    print(f"   operators={form.operators}  focus_hint={form.focus_hint!r}  "
          f"numeric_comparison={form.numeric_comparison}  fell_back={form.fell_back}")
    print()

Q: Is laparoscopic adrenalectomy safe and effective for adrenal masses larger than 7 cm?
   operators=('language_reasoning', 'kg_reasoning', 'numerical_calculation')  focus_hint='laparoscopic adrenalectomy'  numeric_comparison={'attribute': 'size', 'op': '>', 'value': 7.0}  fell_back=False



Q: Was the intervention studied in a population larger than 500 patients?
   operators=('numerical_calculation', 'language_reasoning')  focus_hint='population'  numeric_comparison={'attribute': 'size', 'op': '>', 'value': 500.0}  fell_back=False



Q: What outcome was reported for the largest study of this condition?
   operators=('language_reasoning', 'kg_reasoning')  focus_hint='largest study of this condition'  numeric_comparison=None  fell_back=False



## Full pipeline on real PubMedQA questions

Running every real question in this small sample through `answer_question` end to end and
comparing the predicted verdict against PubMedQA's own real `final_decision` ground truth.

In [3]:
n_correct = 0
for qid, q in data.questions.items():
    answer = answer_question(
        q["question"], data.corpus, doc_ids, matrix, graph, mutual_index,
        embedder=embedder, llm=llm,
    )
    correct = answer.verdict == q["answer"]
    n_correct += correct
    print(f"[{'OK' if correct else 'MISS'}] gold={q['answer']:>5} pred={str(answer.verdict):>5} "
          f"ops={answer.operators_used} | {q['question'][:70]}")

print(f"\n{n_correct}/{len(data.questions)} correct on this sample")

[MISS] gold=  yes pred=maybe ops=('language_reasoning', 'kg_reasoning', 'numerical_calculation') | Is laparoscopic adrenalectomy safe and effective for adrenal masses la


[MISS] gold=  yes pred= None ops=('language_reasoning', 'kg_reasoning') | Malnutrition, a new inducer for arterial calcification in hemodialysis


[MISS] gold=  yes pred=   no ops=('language_reasoning', 'kg_reasoning') | School food policy at Dutch primary schools: room for improvement?


[MISS] gold=  yes pred=maybe ops=('language_reasoning', 'kg_reasoning') | Does laparoscopic antireflux surgery improve quality of life in patien


[MISS] gold=  yes pred=   no ops=('language_reasoning', 'kg_reasoning') | Patient-Controlled Therapy of Breathlessness in Palliative Care: A New


[MISS] gold=  yes pred=   no ops=('language_reasoning', 'kg_reasoning') | Can gingival crevicular blood be relied upon for assessment of blood g


[MISS] gold=  yes pred=   no ops=('language_reasoning', 'kg_reasoning') | Is micro-computed tomography reliable to determine the microstructure 


[MISS] gold=  yes pred=maybe ops=('language_reasoning', 'kg_reasoning') | Does preoperative anemia adversely affect colon and rectal surgery out


[MISS] gold=   no pred= None ops=('language_reasoning', 'kg_reasoning') | Is there a role for fondaparinux in perioperative bridging?


[MISS] gold=  yes pred=maybe ops=('language_reasoning', 'kg_reasoning') | Hypotension in patients with coronary disease: can profound hypotensiv

0/10 correct on this sample


## The numeric operator on the hand-authored population-size question

This only means something if at least one `Population` entity in the graph actually has a
`size` attribute -- whether that happened on this particular real sample is itself part of what
this cell shows.

In [4]:
from indexing.graph_builder import nodes_by_type

population_nodes = nodes_by_type(graph, "Population")
print(f"Population nodes in this graph: {population_nodes}")
for n in population_nodes:
    print(f"  {n}: {graph.nodes[n]['attributes']}")

numeric_question = "Was the intervention studied in a population larger than 500 patients?"
answer = answer_question(
    numeric_question, data.corpus, doc_ids, matrix, graph, mutual_index,
    embedder=embedder, llm=llm,
)
print(f"\nQ: {numeric_question}")
print(f"Operators used: {answer.operators_used}")
print(f"Numeric result: {answer.numeric_result}")
print(f"Verdict: {answer.verdict}")

Population nodes in this graph: ['population-18 consecutive patients', 'hemodialysis patients', 'population-parents', 'population-teachers', 'population-schools', 'population-75', 'population-39', 'population-10', 'population-10490564']
  population-18 consecutive patients: {'size': 18}
  hemodialysis patients: {'size': 68}
  population-parents: {'size': 1429}
  population-teachers: {'size': 72}
  population-schools: {'size': 83}
  population-75: {'size': 75}
  population-39: {'size': 39}
  population-10: {'size': 10}
  population-10490564: {'size': 25}



Q: Was the intervention studied in a population larger than 500 patients?
Operators used: ('numerical_calculation', 'language_reasoning')
Numeric result: NumericResult(values={'population-parents': 1429, 'population-10490564': 25, 'population-75': 75, 'population-18 consecutive patients': 18, 'population-39': 39, 'population-teachers': 72, 'population-10': 10, 'population-schools': 83}, comparison_result=True, extreme_node=None, explanation="size > 500.0 -> True (values seen: {'population-parents': 1429, 'population-10490564': 25, 'population-75': 75, 'population-18 consecutive patients': 18, 'population-39': 39, 'population-teachers': 72, 'population-10': 10, 'population-schools': 83}).")
Verdict: yes


## Observed result

*(filled in after running this notebook against the real, running Ollama instance -- see the
per-question OK/MISS lines and the numeric operator's actual output above.)*